# Mandarin (EN→ZH) Fine-Tuning Pipeline — Sprint 51, Task A

Reproducible pipeline: data validation/versioning/hashing → hardware smoke test →
LoRA/QLoRA fine-tuning → checkpoint load + generation → `candidate_manifest.json`.

**Cantonese is intentionally excluded from this pipeline.**

Run cells top to bottom. Recommended Colab runtime: **T4 GPU** (Runtime → Change runtime type → T4 GPU).


## 1. Environment setup

In [1]:
!nvidia-smi

Wed Sep  9 11:06:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
import shutil, os
os.chdir("/content")
shutil.rmtree("mandarin_pipeline", ignore_errors=True)
os.remove("mandarin_pipeline.zip")

In [ ]:

# ---Upload the provided mandarin_pipeline.zip directly to this Colab session ---
import os, zipfile

ZIP_NAME = "mandarin_pipeline.zip"

if not os.path.exists(ZIP_NAME) and not os.path.exists("mandarin_pipeline"):
    print(f"'{ZIP_NAME}' not found in {os.getcwd()} — opening the upload dialog.")
    print("Select mandarin_pipeline.zip from your computer.")
    from google.colab import files
    uploaded = files.upload()  # blocks until you pick a file
    # in case the uploaded filename differs, pick the first .zip we got
    zips = [f for f in uploaded if f.endswith(".zip")]
    if zips:
        ZIP_NAME = zips[0]

if os.path.exists("mandarin_pipeline"):
    print("mandarin_pipeline/ already present, skipping extraction.")
elif os.path.exists(ZIP_NAME):
    with zipfile.ZipFile(ZIP_NAME) as z:
        z.extractall(".")
    print(f"Extracted {ZIP_NAME}.")
else:
    raise FileNotFoundError(
        f"Still no '{ZIP_NAME}' in {os.getcwd()} after the upload prompt. "
        "Either re-run this cell and upload the file when prompted, or use Option A (git clone) instead."
    )

assert os.path.exists("mandarin_pipeline"), "Extraction did not produce a mandarin_pipeline/ folder — check the zip contents."
%cd mandarin_pipeline
!ls

'mandarin_pipeline.zip' not found in /content — opening the upload dialog.
Select mandarin_pipeline.zip from your computer.


Saving mandarin_pipeline.zip to mandarin_pipeline.zip
Extracted mandarin_pipeline.zip.
/content/mandarin_pipeline
checkpoints  logs	reports		       run_mac_quickstart.sh  tests
config	     notebooks	requirements-cuda.txt  run_pipeline.py
data	     README.md	requirements.txt       src


In [9]:
!pip install -q -r requirements-cuda.txt
# Colab preinstalls an old torchao that is incompatible with newer peft's
# LoRA dispatch check (raises ImportError even though we don't use torchao
# at all for plain LoRA). Removing it makes peft correctly treat it as absent.
!pip uninstall -y torchao -q

## Upload the gold dataset

This pipeline uses the Sprint 50 judge-validation gold export
(`gold_en_cmn.jsonl`, English→Mandarin direction with human-verified
reference translations) as its dataset. Upload it into the
`data/raw/` folder here.


In [10]:
import os
os.makedirs("data/raw", exist_ok=True)

GOLD_FILE = "data/raw/gold_en_cmn.jsonl"
if not os.path.exists(GOLD_FILE):
    print("gold_en_cmn.jsonl not found in data/raw/ — opening the upload dialog.")
    print("Select gold_en_cmn.jsonl from your computer (gold_cmn_en.jsonl is not needed here).")
    from google.colab import files
    uploaded = files.upload()
    for fname, content in uploaded.items():
        target = os.path.join("data/raw", fname)
        with open(target, "wb") as f:
            f.write(content)
    assert os.path.exists(GOLD_FILE), f"Expected {GOLD_FILE} after upload — check the uploaded filename."
else:
    print(f"Found existing {GOLD_FILE}, skipping upload.")

import json
n = sum(1 for _ in open(GOLD_FILE, encoding="utf-8"))
print(f"{GOLD_FILE}: {n} lines")

Found existing data/raw/gold_en_cmn.jsonl, skipping upload.
data/raw/gold_en_cmn.jsonl: 40 lines


## 2. Configure the run

Edit overrides below as needed. Key things to check:
- `run.run_name` — versions this candidate's output folder
- `run.seed` — reproducibility
- `model.base_model` — default is `facebook/nllb-200-distilled-600M` (fits comfortably on a T4)

Note: the default `data.source_type` is `gold_export`, pointed at the
`gold_en_cmn.jsonl` you just uploaded (~40 human-verified examples from the
Sprint 50 judge-validation task). This is a small, high-quality dataset —
fine for producing and validating a candidate checkpoint end-to-end, but
not a substitute for a large corpus in a real production fine-tune. Switch
`data.source_type` to `hf_hub` (see `config/default_config.yaml`) to pull a
much larger EN-ZH corpus (e.g. `Helsinki-NLP/opus-100`) instead.


In [11]:
RUN_NAME = "en-zh-candidate-colab-01"
SEED = 42
DRY_RUN = "true"   # set to "false" once you've confirmed the dry run works end to end

overrides = [
    f"run.run_name={RUN_NAME}",
    f"run.seed={SEED}",
    f"run.dry_run={DRY_RUN}",
]
print(overrides)

['run.run_name=en-zh-candidate-colab-01', 'run.seed=42', 'run.dry_run=true']


## 3. Dry run first (validates the full pipeline wiring cheaply)

This uses a tiny subset of data and 1–2 training steps just to prove:
data prep → hardware smoke test → train → save checkpoint → reload checkpoint → generate → manifest
all work end-to-end **before** committing a full run to it.


In [12]:
import subprocess, sys

cmd = [sys.executable, "run_pipeline.py", "--config", "config/default_config.yaml"]
for o in overrides:
    cmd += ["--set", o]

print(" ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("----- STDERR -----")
    print(result.stderr)
print("Return code:", result.returncode)

/usr/bin/python3 run_pipeline.py --config config/default_config.yaml --set run.run_name=en-zh-candidate-colab-01 --set run.seed=42 --set run.dry_run=true
2026-09-09 11:14:47 | INFO    | pipeline | === Mandarin fine-tuning pipeline | run_name=en-zh-candidate-colab-01 | dry_run=True ===
2026-09-09 11:14:47 | INFO    | pipeline | Resolved config saved to outputs/en-zh-candidate-colab-01/resolved_config.yaml
2026-09-09 11:14:47 | INFO    | pipeline | >>> STAGE 1/5: data_prep
2026-09-09 11:14:47 | INFO    | data_prep | Starting data prep (dry_run=True, max_examples=200)
2026-09-09 11:14:47 | INFO    | data_prep | Loading gold export: data/raw/gold_en_cmn.jsonl (filtering direction=en_to_cmn)
2026-09-09 11:14:47 | INFO    | data_prep | Filtering done: kept=40 dropped_empty=0 dropped_len=0 dropped_dup=0 dropped_langid=0
2026-09-09 11:14:47 | INFO    | data_prep |   split fine_tune_train: 27 examples
2026-09-09 11:14:47 | INFO    | data_prep |   split fine_tune_val: 6 examples
2026-09-09 11:14

## 4. Full run

Once the dry run above completes without errors, switch `DRY_RUN` to `"false"` and re-run.
This is the run that must "complete on Kaggle, Colab or local resources" per the acceptance criteria.


In [13]:
DRY_RUN = "false"
overrides = [
    f"run.run_name={RUN_NAME}",
    f"run.seed={SEED}",
    f"run.dry_run={DRY_RUN}",
]

cmd = [sys.executable, "run_pipeline.py", "--config", "config/default_config.yaml"]
for o in overrides:
    cmd += ["--set", o]

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("----- STDERR -----")
    print(result.stderr)
print("Return code:", result.returncode)

2026-09-09 11:16:53 | INFO    | pipeline | === Mandarin fine-tuning pipeline | run_name=en-zh-candidate-colab-01 | dry_run=False ===
2026-09-09 11:16:53 | INFO    | pipeline | Resolved config saved to outputs/en-zh-candidate-colab-01/resolved_config.yaml
2026-09-09 11:16:53 | INFO    | pipeline | >>> STAGE 1/5: data_prep
2026-09-09 11:16:53 | INFO    | data_prep | Starting data prep (dry_run=False, max_examples=20000)
2026-09-09 11:16:53 | INFO    | data_prep | Loading gold export: data/raw/gold_en_cmn.jsonl (filtering direction=en_to_cmn)
2026-09-09 11:16:53 | INFO    | data_prep | Filtering done: kept=40 dropped_empty=0 dropped_len=0 dropped_dup=0 dropped_langid=0
2026-09-09 11:16:53 | INFO    | data_prep |   split fine_tune_train: 27 examples
2026-09-09 11:16:53 | INFO    | data_prep |   split fine_tune_val: 6 examples
2026-09-09 11:16:53 | INFO    | data_prep |   split judge_calibration: 1 examples
2026-09-09 11:16:53 | INFO    | data_prep |   split prompt_validation: 2 examples
20

### Resuming a full run

If a Colab session disconnects mid-training, just re-run the same cell above with the same
`run.run_name` — `training.resume_from_checkpoint: "auto"` in the config will pick up the
latest checkpoint automatically. You can also skip already-completed stages, e.g.:

```
!python run_pipeline.py --config config/default_config.yaml \
    --set run.run_name=en-zh-candidate-colab-01 --set run.dry_run=false \
    --skip data_prep hardware_check
```


## 5. Inspect the hardware smoke-test decision

In [14]:
import json
report_path = f"outputs/{RUN_NAME}/hardware_report.json"
print(json.dumps(json.load(open(report_path)), indent=2))

{
  "device": "cuda",
  "cuda_available": true,
  "mps_available": false,
  "gpu_name": "Tesla T4",
  "total_gpu_gb": 14.56,
  "free_gpu_gb": 14.46,
  "forward_backward_ok": true,
  "peak_mem_gb_during_test": 3.03,
  "error": null,
  "decision": "lora",
  "decision_reason": "Free GPU memory (14.46GB) is at/above the qlora_memory_threshold_gb (8GB) -> plain LoRA (fp16/bf16) is feasible."
}


## 6. Confirm the checkpoint loads and generates a Mandarin translation

In [15]:
eval_path = f"outputs/{RUN_NAME}/eval_report.json"
eval_report = json.load(open(eval_path, encoding="utf-8"))
print("Checkpoint loaded and generated OK:", eval_report["checkpoint_loaded_and_generated_ok"])
print(f"{eval_report['metric']} score:", eval_report["score"])
print()
for s in eval_report["samples"][:3]:
    print("EN: ", s["source_en"])
    print("REF:", s["reference_zh"])
    print("GEN:", s["candidate_zh"])
    print()

Checkpoint loaded and generated OK: True
chrf score: 31.10377476581252

EN:  Examples of on-site activities include hunting, fishing, photography, bird watching, and visiting parks and studying information about the ecosystem.
REF: 现场活动包括打猎、钓鱼、摄影、观鸟、参观公园和研究生态系统。
GEN: 现场活动的例子包括狩猎,捕鱼,摄影,观鸟,参观公园和研究有关生态系统的信息.

EN:  Although AI has a strong connotation of science fiction, AI forms a very important branch of computer science, dealing with behavior, learning and intelligent adaptation in a machine.
REF: 人工智能有很强的科幻色彩，但它其实是计算机科学非常重要的一个分支，研究的是机器的行为、学习和智能适应。
GEN: 尽管人工智能具有强烈的科幻意义,但人工智能构成了计算机科学的一个非常重要的分支,处理在机器中的行为,学习和智能适应.



## 7. View the candidate manifest

In [16]:
manifest_path = f"outputs/{RUN_NAME}/candidate_manifest.json"
manifest = json.load(open(manifest_path, encoding="utf-8"))
print("candidate_id:", manifest["candidate_id"])
print("data_version:", manifest["data"]["data_version"])
print("method used:", manifest["training"]["method"])
print("checkpoint path:", manifest["checkpoint"]["path"])
print()
print(json.dumps({k: v for k, v in manifest.items() if k not in ("config",)}, indent=2, ensure_ascii=False)[:3000])

candidate_id: en-zh-candidate-colab-01_d69b1564937b7ee4
data_version: d69b1564937b7ee4
method used: lora
checkpoint path: outputs/en-zh-candidate-colab-01/final_checkpoint

{
  "candidate_id": "en-zh-candidate-colab-01_d69b1564937b7ee4",
  "created_at_utc": "2026-09-09T11:18:51.016920+00:00",
  "language_direction": "en-zh",
  "cantonese_excluded": true,
  "seed": 42,
  "data": {
    "data_version": "d69b1564937b7ee4",
    "dataset_name": "data/raw/gold_en_cmn.jsonl",
    "counts": {
      "fine_tune_train": 27,
      "fine_tune_val": 6,
      "judge_calibration": 1,
      "prompt_validation": 2,
      "sealed_test": 4
    },
    "sha256": {
      "fine_tune_train": "26740f27b4f7bb57e71d83686802fcb67b9564eab15a45b45869597db9cf2b9c",
      "fine_tune_val": "99e84d94488e59bb7a15af4d3549b07c9a913c8d64983d7c6f2656fbc6e31b01",
      "judge_calibration": "7d4ca9562e2b90ca60431c88859f10c8a2a040061e4e3c2664b29847f1d7012d",
      "prompt_validation": "649875ca32204055b7a8886a506aadd94a532421349

## 8. Package outputs for submission

Zips config, manifests, logs, report, and the checkpoint together so you can download
them or push to Drive / GitHub Releases (checkpoints are usually too large for a normal git push).


In [17]:
import shutil
zip_path = shutil.make_archive(f"{RUN_NAME}_submission", "zip", f"outputs/{RUN_NAME}")
print("Packaged:", zip_path)

from google.colab import files
files.download(zip_path)

Packaged: /content/mandarin_pipeline/en-zh-candidate-colab-01_submission.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. (Optional) Push checkpoint + manifest to Hugging Face Hub or Google Drive

Large checkpoint files should not go into git directly. Two common options:

```python
# Option A: Google Drive
from google.colab import drive
drive.mount('/content/drive')
shutil.copytree(f"outputs/{RUN_NAME}", f"/content/drive/MyDrive/{RUN_NAME}", dirs_exist_ok=True)
```

```python
# Option B: Hugging Face Hub (requires `huggingface_hub` + a token with write access)
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(repo_id="<your-username>/en-zh-candidate-01", exist_ok=True)
api.upload_folder(folder_path=f"outputs/{RUN_NAME}/final_checkpoint",
                   repo_id="<your-username>/en-zh-candidate-01")
```

Record whichever download/hub path you use as the **"loadable candidate or download path"**
deliverable, alongside the exact GitHub commit link for the code.
